##Libraries

In [2]:
%%capture
%pip install langchain
%pip install langchain-community
%pip install langchain-huggingface
%pip install langchain-groq
%pip install faiss-cpu
%pip install sentence-transformers
%pip install python-dotenv

In [3]:
import os
from google.colab import userdata

##Load HuggingFace Embeddings

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##Create Dataset

In [5]:
faqs = [
    "What is FAISS? FAISS is a vector database library developed by Meta.",
    "What is RAG? Retrieval Augmented Generation combines retrieval and LLM generation.",
    "What is LangChain? LangChain helps build LLM applications.",
    "What is Groq? Groq provides ultra-fast inference for LLMs.",
    "What are embeddings? Embeddings convert text into vectors.",
    "What is cosine similarity? It measures similarity between vectors.",
    "What is a vector database? It stores embeddings for semantic search.",
    "What is chunking? Splitting text/documents into smaller pieces.",
    "What is reranking? Improving retrieval quality after search.",
    "What is hallucination? When an LLM generates incorrect information out of context."
]

##FAISS

In [6]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(
    faqs,
    embedding_model
)

/tmp/ipykernel_12074/2565758747.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


##Groq LLM

In [8]:
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

##Similarity Search Function

In [10]:
def similarity_search(query):
    docs = vector_store.similarity_search_with_score(
        query,
        k=2
    )

    return docs

##Generation Function

In [11]:
def answer_question(query):

    results = similarity_search(query)

    context = "\n".join(
        [doc.page_content for doc, score in results]
    )

    prompt = f"""
You are a FAQ assistant.

Use only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content, results

##Build CLI Interface

In [12]:
while True:

    query = input("\nAsk Question: ")

    if query.lower() == "exit":
        break

    answer, docs = answer_question(query)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved FAQs:")

    for doc, score in docs:
        print(f"\nScore: {score:.4f}")
        print(doc.page_content)


Ask Question: What is vector database

Answer:
A vector database stores embeddings for semantic search.

Retrieved FAQs:

Score: 0.3477
What is a vector database? It stores embeddings for semantic search.

Score: 1.0684
What is FAISS? FAISS is a vector database library developed by Meta.

Ask Question: What is grq

Answer:
Groq provides ultra-fast inference for LLMs.

Retrieved FAQs:

Score: 0.7546
What is Groq? Groq provides ultra-fast inference for LLMs.

Score: 1.5796
What is RAG? Retrieval Augmented Generation combines retrieval and LLM generation.

Ask Question: what is google

Answer:
Google is not mentioned in the provided context. The context only discusses vector databases and reranking, but does not provide information about Google.

Retrieved FAQs:

Score: 1.4975
What is a vector database? It stores embeddings for semantic search.

Score: 1.4992
What is reranking? Improving retrieval quality after search.

Ask Question: google

Answer:
Your input is not a question, it seems